# 1IEP named preparation: rigid and explicit flexible controls

Owning DockingMT issues: #17, #6 and #5. Original twelve searches are inspected, not rerun. The bound-like rigid starting conformer is a favorable control. Both preparations and the external receptor remain scientifically unassessed.

In [1]:
from pathlib import Path
import gzip, hashlib, json, sys
import numpy as np
import molsysmt as msm
import pyunitwizard as puw
import dockingmt as dmt
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
sys.path.insert(0, str(root))
from devtools.qualify_1iep_flexibility import prepare_case
path = root / 'devguide/validation/data/1iep_flexibility/audit_2026-10-07.json.gz'
raw = gzip.decompress(path.read_bytes())
record = json.loads(raw)
assert len(record['redocking_runs']) == 12
assert record['qualified_provider_commit'] == '5bd893c85fe8d211663b2b1f865f5f1d2c382a90'
for name, digest in record['input_sha256'].items():
    assert hashlib.sha256((root / 'tests/data/vina_torsions' / name).read_bytes()).hexdigest() == digest
print('Python:', sys.version)
print('Original JSON SHA-256:', hashlib.sha256(raw).hexdigest())
print('Preserved searches:', len(record['redocking_runs']))

Python: 3.14.7 | packaged by conda-forge | (main, Sep  2 2026, 21:08:32) [GCC 15.3.0]
Original JSON SHA-256: d581f86bf8682a0b045dbcc73e0c8e1979161a17a16d8f109e20da1c991df9b2
Preserved searches: 12


In [2]:
print('variant  effort seed poses first-heavy closest-heavy first-full score')
for run in record['redocking_runs']:
    p = run['result']['protocol_info']['parameters']
    heavy = run['heavy_atom_metrics']
    values = [row['heavy_atom_positional_rmsd_angstrom'] for row in heavy]
    print(f"{run['variant']:8s} {p['exhaustiveness']:6d} {p['seed']:4d} {len(heavy):5d} {values[0]:11.4f} {min(values):13.4f} {run['evaluation']['first_pose']['rmsd']:10.4f} {heavy[0]['vina_score_kcal_per_mol']:7.3f}")
    assert dmt.verify_captured_inputs(run['result']['provenance']['backend_artifacts'])
print('RMSD: angstrom, positional, no alignment/symmetry correction; score: Vina kcal/mol.')

variant  effort seed poses first-heavy closest-heavy first-full score
rigid         1    7     1     13.2896       13.2896    13.2160  -7.029
flexible      1    7     3     12.3453       12.3453    12.3817 -10.899
rigid         1   42     1     13.2777       13.2777    13.2061  -6.996
flexible      1   42     1     12.3663       12.3663    12.4005 -10.779
rigid         1 2026     1      0.2164        0.2164     0.2192 -18.377
flexible      1 2026     3     12.3183       12.1256    12.3563 -10.888
rigid         8    7     1      0.2195        0.2195     0.2221 -18.359
flexible      8    7     3      0.2928        0.2928     0.3113 -13.302
rigid         8   42     1      0.2082        0.2082     0.2109 -18.385
flexible      8   42     5     12.3349       12.0659    12.3710 -10.827
rigid         8 2026     1      0.1945        0.1945     0.1965 -18.378
flexible      8 2026     5     12.3183       12.0296    12.3563 -10.888
RMSD: angstrom, positional, no alignment/symmetry correction; scor

In [3]:
source, preparations, audit = prepare_case()
run = next(r for r in record['redocking_runs'] if r['variant'] == 'flexible')
restored = dmt.DockingResult.from_dict(run['result'])
with puw.context(standard_units=['pm', 'fs', 'K', 'mole', 'dalton', 'e', 'kJ/mol', 'radians']):
    observed = dmt.evaluate_redocking(restored, source, rmsd_cutoff=puw.quantity(250, 'pm'))
assert observed['criterion'] == run['evaluation']['criterion']
np.testing.assert_allclose([r['rmsd'] for r in observed['poses']], [r['rmsd'] for r in run['evaluation']['poses']], rtol=0, atol=1e-12)
assert audit['source_unchanged']
print('Saved-result source identities and RMSDs preserved under pm/fs.')

Saved-result source identities and RMSDs preserved under pm/fs.
